In [12]:
# ============================================================
# D2 — Branch C: Normalised representation
# 0. Imports and frozen experimental configuration
# ============================================================
!pip -q install pymupdf

import json
import hashlib
import re
import sys
import platform
import unicodedata
import fitz

from pathlib import Path
from datetime import datetime
from google.colab import files

DOCUMENT_ID = "D2"
DOCUMENT_NAME = "VINCI Consolidated Income Statement 2024"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

EXPECTED_SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667"
EXPECTED_PAGE_COUNT = 1
EXPECTED_RECORD_COUNT = 22

EXPECTED_FIELDS = [
    "Line Item",
    "Unit",
    "Value 2024",
    "Value 2023"
]

NUMERIC_FIELDS = [
    "Value 2024",
    "Value 2023"
]

ALLOWED_UNITS = [
    "EUR millions",
    "EUR"
]

OUTPUT_DIR = Path("outputs_D2_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Parent branch:", PARENT_BRANCH)


Document: D2
Branch: C
Parent branch: B


In [13]:
# ------------------------------------------------------------
# 1. Upload the original PDF and the two required Branch B artefacts
# ------------------------------------------------------------
# Upload exactly:
#   1) original D2 PDF
#   2) D2_branch_B_structural_markdown.md
#   3) D2_branch_B_conversion_integrity.json

uploaded = files.upload()
names = list(uploaded.keys())

pdf_files = [Path(f) for f in names if f.lower().endswith(".pdf")]
md_files = [Path(f) for f in names if f.lower().endswith(".md")]
json_files = [Path(f) for f in names if f.lower().endswith(".json")]

if len(pdf_files) != 1 or len(md_files) != 1 or len(json_files) != 1:
    raise ValueError(
        "Upload exactly one PDF, one Branch B Markdown file, "
        "and one Branch B conversion-integrity JSON file."
    )

SOURCE_FILE = pdf_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_FILE.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D2_branch_B_conversion_integrity.json to D2_branch_B_conversion_integrity.json
Saving D2_branch_B_structural_markdown.md to D2_branch_B_structural_markdown.md
Saving D2 - 2024-vinci-Income-Statement.pdf to D2 - 2024-vinci-Income-Statement.pdf
Source: D2 - 2024-vinci-Income-Statement.pdf
Branch B representation: D2_branch_B_structural_markdown.md
Branch B integrity: D2_branch_B_conversion_integrity.json


In [14]:
# ------------------------------------------------------------
# 2. Verify frozen source identity and Branch B parent integrity
# ------------------------------------------------------------
def sha256_file(path, chunk_size=8192):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

if SOURCE_FILE.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError("Unexpected D2 source format.")

SOURCE_SHA256 = sha256_file(SOURCE_FILE)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if not SOURCE_HASH_MATCH:
    raise ValueError("Uploaded PDF does not match the frozen D2 source identity.")

with open(BRANCH_B_CHECK_PATH, "r", encoding="utf-8") as f:
    branch_b_check = json.load(f)

if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError("The Branch B integrity file belongs to another document.")

if branch_b_check.get("branch") != "B":
    raise ValueError("The uploaded integrity file is not from Branch B.")

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError("Branch B was generated from a different source identity.")

if not branch_b_check.get("conversion_integrity_passed", False):
    raise ValueError("Branch B parent representation did not pass conversion integrity.")

SOURCE_B_MARKDOWN = BRANCH_B_REPRESENTATION_PATH.read_text(encoding="utf-8")

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError("Uploaded Branch B Markdown is empty.")

SOURCE_B_REPRESENTATION_SHA256 = sha256_text(SOURCE_B_MARKDOWN)

print("Frozen source identity verified.")
print("Uploaded Branch B representation SHA-256:", SOURCE_B_REPRESENTATION_SHA256)


Frozen source identity verified.
Uploaded Branch B representation SHA-256: 4b3344e3ba0da62330c724aea89fe180e32877a349669d95155a22d7d0bb62de


In [15]:
# ------------------------------------------------------------
# 3. Reproduce the exact Branch B structural representation
# ------------------------------------------------------------
# This repeats the deterministic Branch B structural-conversion logic
# only to verify lineage. No Branch C normalisation is applied here.

pdf_document = fitz.open(SOURCE_FILE)

if len(pdf_document) != EXPECTED_PAGE_COUNT:
    raise ValueError("Unexpected D2 page count.")

page = pdf_document[0]

if not page.get_text("text").strip():
    raise ValueError("D2 is expected to contain machine-readable text.")

raw_blocks = page.get_text("blocks")

ordered_blocks = sorted(
    raw_blocks,
    key=lambda block: (
        round(block[1], 3),
        round(block[0], 3)
    )
)

block_audit = []

for i, block in enumerate(ordered_blocks):
    text = block[4].strip()

    if text:
        block_audit.append({
            "block_index": i,
            "x0": float(block[0]),
            "y0": float(block[1]),
            "x1": float(block[2]),
            "y1": float(block[3]),
            "text": text
        })

def nonempty_lines(text):
    return [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

header_block = None
footnote_block = None
row_blocks = []

for item in block_audit:
    lines = nonempty_lines(item["text"])

    if (
        "(in € millions)" in item["text"]
        and "2024" in item["text"]
        and "2023" in item["text"]
    ):
        header_block = item
        continue

    if "(*) Excluding concession subsidiaries" in item["text"]:
        footnote_block = item
        continue

    if len(lines) == 3:
        row_blocks.append(item)

if header_block is None or footnote_block is None:
    raise ValueError("Could not reproduce the Branch B table structure.")

if len(row_blocks) != EXPECTED_RECORD_COUNT:
    raise ValueError(
        f"Reproduced Branch B row count is {len(row_blocks)}, "
        f"expected {EXPECTED_RECORD_COUNT}."
    )

converted_rows = []

for item in row_blocks:
    lines = nonempty_lines(item["text"])
    converted_rows.append({
        "Line Item": lines[0],
        "2024": lines[1],
        "2023": lines[2]
    })

def escape_markdown_cell(value):
    return (
        str(value)
        .replace("\\", "\\\\")
        .replace("|", "\\|")
        .replace("\n", "<br>")
    )

header_lines = nonempty_lines(header_block["text"])
table_unit = header_lines[0]

footnote_text = " ".join(
    nonempty_lines(footnote_block["text"])
)

markdown_rows = [
    "| Line Item | 2024 | 2023 |",
    "| --- | ---: | ---: |"
]

for row in converted_rows:
    markdown_rows.append(
        "| "
        + " | ".join([
            escape_markdown_cell(row["Line Item"]),
            escape_markdown_cell(row["2024"]),
            escape_markdown_cell(row["2023"])
        ])
        + " |"
    )

REPRODUCED_BRANCH_B_MARKDOWN = (
    "# Consolidated financial statements\n\n"
    "## Consolidated income statement\n\n"
    f"{table_unit}\n\n"
    + "\n".join(markdown_rows)
    + "\n\n"
    + footnote_text
    + "\n"
)

REPRODUCED_B_SHA256 = sha256_text(REPRODUCED_BRANCH_B_MARKDOWN)


In [16]:
# ------------------------------------------------------------
# 4. Verify exact Branch B parent equivalence
# ------------------------------------------------------------
PARENT_EQUIVALENCE_PASSED = (
    REPRODUCED_BRANCH_B_MARKDOWN == SOURCE_B_MARKDOWN
)

parent_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,
    "branch_B_conversion_integrity_passed":
        bool(branch_b_check.get("conversion_integrity_passed", False)),
    "uploaded_branch_B_sha256": SOURCE_B_REPRESENTATION_SHA256,
    "reproduced_branch_B_sha256": REPRODUCED_B_SHA256,
    "branch_B_representation_exactly_reproduced":
        PARENT_EQUIVALENCE_PASSED,
    "reproduced_record_count": len(converted_rows),
    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}

PARENT_CHECK_PATH = OUTPUT_DIR / "D2_branch_C_parent_B_equivalence_check.json"
PARENT_CHECK_PATH.write_text(
    json.dumps(parent_check, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(parent_check, indent=2, ensure_ascii=False))

if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "The uploaded Branch B Markdown does not exactly match the "
        "structural representation reproduced from the frozen D2 source."
    )


{
  "document_id": "D2",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667",
  "branch_B_conversion_integrity_passed": true,
  "uploaded_branch_B_sha256": "4b3344e3ba0da62330c724aea89fe180e32877a349669d95155a22d7d0bb62de",
  "reproduced_branch_B_sha256": "4b3344e3ba0da62330c724aea89fe180e32877a349669d95155a22d7d0bb62de",
  "branch_B_representation_exactly_reproduced": true,
  "reproduced_record_count": 22,
  "parent_equivalence_passed": true
}


In [17]:
# ------------------------------------------------------------
# 5. Define deterministic Branch C normalisation
# ------------------------------------------------------------

UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f", "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "–": "-",
    "—": "-",
    "−": "-",
    "‐": "-"
}


def normalise_unicode(text):
    """
    Apply Unicode compatibility normalisation without changing
    document meaning.
    """
    return unicodedata.normalize("NFKC", text)


def normalise_unicode_spaces(text):
    """
    Replace non-standard Unicode space characters with ordinary spaces.
    Newline characters are deliberately preserved.
    """
    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(character, " ")

    return text


def normalise_apostrophes(text):
    """
    Standardise typographic apostrophe variants.
    """
    for original, replacement in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(original, replacement)

    return text


def normalise_dashes(text):
    """
    Standardise dash/minus glyph variants to an ASCII hyphen.
    """
    for original, replacement in DASH_REPLACEMENTS.items():
        text = text.replace(original, replacement)

    return text


def normalise_line_whitespace(text):
    """
    Collapse spaces and tabs inside individual lines.

    IMPORTANT:
    Line boundaries are preserved because they encode Markdown
    table structure and separation between the table and footnote.
    """
    lines = []

    for line in text.splitlines():

        line = re.sub(
            r"[ \t]+",
            " ",
            line
        ).strip()

        lines.append(line)

    return "\n".join(lines)


def normalise_blank_lines(text):
    """
    Standardise excessive blank lines while preserving structural
    separation between document components.
    """
    return (
        re.sub(
            r"\n{3,}",
            "\n\n",
            text
        )
        .strip()
        + "\n"
    )


def normalise_unit_header(text):
    """
    Standardise the table-level unit notation.

    Examples:
        (in € millions)
        (in EUR millions)

    become:
        (in EUR millions)

    This changes notation only, not the financial values or their unit.
    """

    patterns = [
        r"\(\s*in\s*€\s*millions\s*\)",
        r"\(\s*in\s+EUR\s+millions\s*\)"
    ]

    for pattern in patterns:

        text = re.sub(
            pattern,
            "(in EUR millions)",
            text,
            flags=re.IGNORECASE
        )

    return text


def normalise_footnote_marker_spacing(text):
    """
    Standardise spacing around the (*) marker WITHOUT consuming
    newline characters.

    The previous implementation used \\s*, which also matches newlines
    and could join the footnote to the final Markdown table row.

    Only horizontal whitespace (spaces/tabs) is normalised here.
    """

    return re.sub(
        r"[ \t]*\([ \t]*\*[ \t]*\)",
        " (*)",
        text
    )


def normalise_representation(text):
    """
    Apply the deterministic Branch C normalisation pipeline.

    Operations are representation-level only:
    - Unicode normalisation
    - Unicode-space standardisation
    - apostrophe standardisation
    - dash standardisation
    - intra-line whitespace normalisation
    - unit notation standardisation
    - footnote-marker spacing standardisation
    - blank-line standardisation

    No source values are inferred, calculated, rounded, repaired,
    or semantically rewritten.
    """

    text = normalise_unicode(text)

    text = normalise_unicode_spaces(text)

    text = normalise_apostrophes(text)

    text = normalise_dashes(text)

    text = normalise_line_whitespace(text)

    text = normalise_unit_header(text)

    text = normalise_footnote_marker_spacing(text)

    text = normalise_blank_lines(text)

    return text

In [18]:
# ------------------------------------------------------------
# 6. Generate the Branch C representation
# ------------------------------------------------------------
NORMALISED_MARKDOWN = normalise_representation(
    SOURCE_B_MARKDOWN
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError("Branch C normalisation produced an empty representation.")

print("Branch B characters:", len(SOURCE_B_MARKDOWN))
print("Branch C characters:", len(NORMALISED_MARKDOWN))
print("\nBranch C representation:\n")
print(NORMALISED_MARKDOWN)


Branch B characters: 1406
Branch C characters: 1409

Branch C representation:

# Consolidated financial statements

## Consolidated income statement

(in EUR millions)

| Line Item | 2024 | 2023 |
| --- | ---: | ---: |
| Revenue (*) | 71,623 | 68,838 |
| Concession subsidiaries' revenue derived from works carried out by non-Group companies | 837 | 780 |
| Total revenue | 72,459 | 69,619 |
| Revenue from ancillary activities | 308 | 267 |
| Operating expenses | (63,770) | (61,529) |
| Operating income from ordinary activities | 8,997 | 8,357 |
| Share-based payments (IFRS 2) | (462) | (360) |
| Profit/(loss) of companies accounted for under the equity method | 219 | 111 |
| Other recurring operating items | 97 | 68 |
| Recurring operating income | 8,850 | 8,175 |
| Non-recurring operating items | (68) | (105) |
| Operating income | 8,783 | 8,071 |
| Cost of gross financial debt | (1,785) | (1,363) |
| Financial income from cash investments | 595 | 469 |
| Cost of net financial debt | (1

In [19]:
# ------------------------------------------------------------
# 7. Verify Branch C normalisation integrity
# ------------------------------------------------------------
def extract_numeric_tokens(text):
    pattern = r"\(?-?\d[\d,]*(?:\.\d+)?\)?"
    return re.findall(pattern, text)

source_numeric_tokens = extract_numeric_tokens(SOURCE_B_MARKDOWN)
normalised_numeric_tokens = extract_numeric_tokens(NORMALISED_MARKDOWN)

numeric_tokens_preserved = (
    source_numeric_tokens == normalised_numeric_tokens
)

EXPECTED_CONTENT_MARKERS = [
    "Revenue",
    "Operating income",
    "Net income",
    "Basic earnings per share",
    "Diluted earnings per share",
    "2024",
    "2023"
]

content_marker_results = {
    marker: marker.casefold() in NORMALISED_MARKDOWN.casefold()
    for marker in EXPECTED_CONTENT_MARKERS
}

# Verify the 22 row identities remain represented in the same order.
def extract_markdown_line_items(markdown_text):
    items = []
    for line in markdown_text.splitlines():
        if not line.startswith("| ") or line.startswith("| ---"):
            continue

        parts = [p.strip() for p in line.strip("|").split("|")]

        if len(parts) != 3:
            continue

        if parts[0] == "Line Item":
            continue

        items.append(parts[0].replace("\\|", "|").replace("\\\\", "\\"))

    return items

branch_b_line_items = extract_markdown_line_items(SOURCE_B_MARKDOWN)
branch_c_line_items = extract_markdown_line_items(NORMALISED_MARKDOWN)

# Because Branch C is allowed to standardise punctuation and Unicode,
# compare identities after applying the same normalisation to both sides.
canonical_b_line_items = [
    normalise_representation(item).strip()
    for item in branch_b_line_items
]
canonical_c_line_items = [
    normalise_representation(item).strip()
    for item in branch_c_line_items
]

observation_identity_and_order_preserved = (
    canonical_b_line_items == canonical_c_line_items
)

normalisation_integrity_passed = bool(
    PARENT_EQUIVALENCE_PASSED
    and len(branch_c_line_items) == EXPECTED_RECORD_COUNT
    and numeric_tokens_preserved
    and all(content_marker_results.values())
    and observation_identity_and_order_preserved
)

normalisation_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,
    "parent_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "represented_line_item_count": len(branch_c_line_items),
    "record_count_preserved":
        len(branch_c_line_items) == EXPECTED_RECORD_COUNT,
    "observation_identity_and_order_preserved":
        observation_identity_and_order_preserved,
    "numeric_tokens_preserved":
        numeric_tokens_preserved,
    "expected_content_markers":
        content_marker_results,
    "all_expected_content_markers_present":
        all(content_marker_results.values()),
    "unicode_nfkc_normalisation_applied": True,
    "unicode_space_standardisation_applied": True,
    "apostrophe_standardisation_applied": True,
    "dash_standardisation_applied": True,
    "whitespace_normalisation_applied": True,
    "unit_header_standardisation_applied": True,
    "footnote_marker_spacing_standardisation_applied": True,
    "semantic_label_rewriting_applied": False,
    "accounting_terminology_harmonisation_applied": False,
    "manual_correction_applied": False,
    "missing_content_reconstruction_applied": False,
    "value_modification_applied": False,
    "value_rounding_applied": False,
    "derived_calculation_applied": False,
    "reference_values_used_for_transformation": False,
    "normalisation_integrity_passed":
        normalisation_integrity_passed
}

NORMALISATION_CHECK_PATH = OUTPUT_DIR / "D2_branch_C_normalisation_check.json"
NORMALISATION_CHECK_PATH.write_text(
    json.dumps(normalisation_check, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(normalisation_check, indent=2, ensure_ascii=False))

if not normalisation_integrity_passed:
    raise ValueError("D2 Branch C normalisation-integrity checks failed.")


{
  "document_id": "D2",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "expected_record_count": 22,
  "represented_line_item_count": 22,
  "record_count_preserved": true,
  "observation_identity_and_order_preserved": true,
  "numeric_tokens_preserved": true,
  "expected_content_markers": {
    "Revenue": true,
    "Operating income": true,
    "Net income": true,
    "Basic earnings per share": true,
    "Diluted earnings per share": true,
    "2024": true,
    "2023": true
  },
  "all_expected_content_markers_present": true,
  "unicode_nfkc_normalisation_applied": true,
  "unicode_space_standardisation_applied": true,
  "apostrophe_standardisation_applied": true,
  "dash_standardisation_applied": true,
  "whitespace_normalisation_applied": true,
  "unit_header_standardisation_applied": true,
  "footnote_marker_spacing_standardisation_applied": true,
  "semantic_label_rewriting_applied": false,
  "accounting_terminology_harmonisation_applied": false,
 

In [20]:
# ------------------------------------------------------------
# 8. Save the normalised representation
# ------------------------------------------------------------
REPRESENTATION_PATH = OUTPUT_DIR / "D2_branch_C_normalised_markdown.md"
REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(REPRESENTATION_PATH)

print("Saved:", REPRESENTATION_PATH.name)
print("Representation SHA-256:", REPRESENTATION_SHA256)


Saved: D2_branch_C_normalised_markdown.md
Representation SHA-256: cc5ca221b7a5af52fe977137c5ae4f4773cc7d0243719ea18be80f4b4d609143


In [21]:
# ------------------------------------------------------------
# 9. Define the fixed extraction schema
# ------------------------------------------------------------
EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level":
        "consolidated_income_statement_line_item",
    "fields": {
        "Line Item": {
            "type": ["string", "null"],
            "description":
                "Exact visible income-statement row label"
        },
        "Unit": {
            "type": ["string", "null"],
            "allowed_values": ALLOWED_UNITS
        },
        "Value 2024": {
            "type": ["number", "null"],
            "description":
                "Reported numerical value for 2024"
        },
        "Value 2023": {
            "type": ["number", "null"],
            "description":
                "Reported numerical value for 2023"
        }
    },
    "expected_output_structure": {
        "document_id": DOCUMENT_ID,
        "branch": BRANCH,
        "records": [
            {
                "Line Item": None,
                "Unit": None,
                "Value 2024": None,
                "Value 2023": None
            }
        ]
    }
}

print(json.dumps(EXTRACTION_SCHEMA, indent=2, ensure_ascii=False))


{
  "document_id": "D2",
  "record_level": "consolidated_income_statement_line_item",
  "fields": {
    "Line Item": {
      "type": [
        "string",
        "null"
      ],
      "description": "Exact visible income-statement row label"
    },
    "Unit": {
      "type": [
        "string",
        "null"
      ],
      "allowed_values": [
        "EUR millions",
        "EUR"
      ]
    },
    "Value 2024": {
      "type": [
        "number",
        "null"
      ],
      "description": "Reported numerical value for 2024"
    },
    "Value 2023": {
      "type": [
        "number",
        "null"
      ],
      "description": "Reported numerical value for 2023"
    }
  },
  "expected_output_structure": {
    "document_id": "D2",
    "branch": "C",
    "records": [
      {
        "Line Item": null,
        "Unit": null,
        "Value 2024": null,
        "Value 2023": null
      }
    ]
  }
}


In [22]:
# ------------------------------------------------------------
# 10. Define the controlled Branch C extraction task
# ------------------------------------------------------------
# This mirrors Branch B instruction strength.
# The expected record count is NOT disclosed to the model.

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every line-item observation from the consolidated income
statement contained in the attached deterministically normalised
Markdown document.

Return one record for every visible income-statement line item.

For each record, extract:

- Line Item
- Unit
- Value 2024
- Value 2023

Extraction rules:

- Treat the attached deterministically normalised Markdown document as
  the only source of information.
- Extract only information explicitly supported by the document.
- Preserve each line-item label exactly as represented in the source
  representation, including footnote markers and unit text contained
  in the label.
- Preserve the association between each line item and its corresponding
  2024 and 2023 values.
- Use "EUR millions" for values governed by the table-level unit
  "(in EUR millions)".
- Use "EUR" for the two earnings-per-share observations.
- Convert financial values shown in parentheses into negative numerical
  values.
- Return Value 2024 and Value 2023 as numerical values.
- Do not calculate, infer, reconstruct, aggregate, correct or invent
  any value.
- Use null only when a requested value is not available.
- Do not include the table title, year headers, unit header or footnote
  explanation as separate records.
- Verify that every visible income-statement line item has been processed.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
""".strip()

EXPECTED_OUTPUT_STRUCTURE = EXTRACTION_SCHEMA["expected_output_structure"]

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

The deterministically normalised Markdown document is attached as the
extraction source.

Return only the JSON object.
""".strip()

PROMPT_PATH = OUTPUT_DIR / "D2_branch_C_prompt.txt"
PROMPT_PATH.write_text(FULL_PROMPT, encoding="utf-8")
PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print(FULL_PROMPT)
print("Prompt SHA-256:", PROMPT_SHA256)


You are an information extraction assistant.

Extract every line-item observation from the consolidated income
statement contained in the attached deterministically normalised
Markdown document.

Return one record for every visible income-statement line item.

For each record, extract:

- Line Item
- Unit
- Value 2024
- Value 2023

Extraction rules:

- Treat the attached deterministically normalised Markdown document as
  the only source of information.
- Extract only information explicitly supported by the document.
- Preserve each line-item label exactly as represented in the source
  representation, including footnote markers and unit text contained
  in the label.
- Preserve the association between each line item and its corresponding
  2024 and 2023 values.
- Use "EUR millions" for values governed by the table-level unit
  "(in EUR millions)".
- Use "EUR" for the two earnings-per-share observations.
- Convert financial values shown in parentheses into negative numerical
  values.


In [23]:
# ------------------------------------------------------------
# 11. Create representation and experiment metadata
# ------------------------------------------------------------
representation_metadata = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,
    "source_file": SOURCE_FILE.name,
    "source_sha256": SOURCE_SHA256,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "representation_type":
        "Deterministically normalised structural Markdown",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "structural_conversion_inherited_from_branch_B": True,
    "normalisation_applied": True,
    "normalisation_operations": [
        "Unicode NFKC normalisation",
        "Unicode-space standardisation",
        "apostrophe standardisation",
        "dash standardisation",
        "intra-line whitespace cleanup",
        "blank-line standardisation",
        "table-level unit-header notation standardisation",
        "footnote-marker spacing standardisation"
    ],
    "semantic_label_rewriting_applied": False,
    "accounting_terminology_harmonisation_applied": False,
    "manual_correction_applied": False,
    "missing_content_reconstruction_applied": False,
    "value_modification_applied": False,
    "value_rounding_applied": False,
    "derived_calculation_applied": False,
    "reference_values_used_for_transformation": False,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"]
}

REP_METADATA_PATH = OUTPUT_DIR / "D2_branch_C_representation_metadata.json"
REP_METADATA_PATH.write_text(
    json.dumps(representation_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

experiment_metadata = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "input_representation":
        "Deterministically normalised structural Markdown",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "normalisation_applied": True,
    "ocr_applied": False,
    "reference_values_disclosed_to_model": False,
    "expected_record_count_disclosed_to_model": False,
    "manual_response_repair_permitted": False,
    "expected_output_format": "JSON",
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "execution_environment": "Independent ChatGPT conversation",
    "model": "GPT-5.5",
    "created_at": datetime.now().isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "validation_status":
        "Pending Stage 4 Branch C validation against the fixed Stage 1 "
        "reference dataset using Branch A-frozen comparison rules"
}

METADATA_PATH = OUTPUT_DIR / "D2_branch_C_experiment_metadata.json"
METADATA_PATH.write_text(
    json.dumps(experiment_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(experiment_metadata, indent=2, ensure_ascii=False))


{
  "document_id": "D2",
  "document_name": "VINCI Consolidated Income Statement 2024",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_sha256": "6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667",
  "source_verified": true,
  "input_representation": "Deterministically normalised structural Markdown",
  "representation_file": "D2_branch_C_normalised_markdown.md",
  "representation_sha256": "cc5ca221b7a5af52fe977137c5ae4f4773cc7d0243719ea18be80f4b4d609143",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "normalisation_applied": true,
  "ocr_applied": false,
  "reference_values_disclosed_to_model": false,
  "expected_record_count_disclosed_to_model": false,
  "manual_response_repair_permitted": false,
  "expected_output_format": "JSON",
  "prompt_file": "D2_branch_C_prompt.txt",
  "prompt_sha256": "efe86633dfb

In [24]:
# ------------------------------------------------------------
# 12. Final pre-extraction control check
# ------------------------------------------------------------
precheck = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_identity_verified": SOURCE_HASH_MATCH,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],
    "representation_exists": REPRESENTATION_PATH.exists(),
    "prompt_exists": PROMPT_PATH.exists(),
    "expected_record_count_disclosed_to_model": False,
    "reference_values_used_for_transformation": False,
    "ready_for_independent_llm_execution": bool(
        SOURCE_HASH_MATCH
        and PARENT_EQUIVALENCE_PASSED
        and normalisation_check["normalisation_integrity_passed"]
        and REPRESENTATION_PATH.exists()
        and PROMPT_PATH.exists()
    )
}

PRECHECK_PATH = OUTPUT_DIR / "D2_branch_C_pre_extraction_check.json"
PRECHECK_PATH.write_text(
    json.dumps(precheck, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(precheck, indent=2, ensure_ascii=False))

if not precheck["ready_for_independent_llm_execution"]:
    raise ValueError("D2 Branch C is not ready for independent LLM execution.")


{
  "document_id": "D2",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [25]:
# ------------------------------------------------------------
# 13. Download pre-extraction Branch C artefacts
# ------------------------------------------------------------
for p in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH
]:
    files.download(p)

print(
    "\nIndependent execution instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D2_branch_C_normalised_markdown.md.\n"
    "3. Submit the exact contents of D2_branch_C_prompt.txt once.\n"
    "4. Do not upload the original PDF, Branch B artefacts, or Stage 1 reference values.\n"
    "5. Do not manually correct, regenerate or repair the response.\n"
    "6. Save the complete response exactly as returned in a plain-text file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent execution instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D2_branch_C_normalised_markdown.md.
3. Submit the exact contents of D2_branch_C_prompt.txt once.
4. Do not upload the original PDF, Branch B artefacts, or Stage 1 reference values.
5. Do not manually correct, regenerate or repair the response.
6. Save the complete response exactly as returned in a plain-text file.


In [26]:
# ------------------------------------------------------------
# 14. Upload and preserve the complete raw Branch C response
# ------------------------------------------------------------
uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError("Upload exactly one complete raw Branch C response file.")

RAW_RESPONSE_SOURCE = Path(next(iter(uploaded_response)))
RAW_RESPONSE_TEXT = RAW_RESPONSE_SOURCE.read_text(encoding="utf-8")

RAW_RESPONSE_PATH = OUTPUT_DIR / "D2_branch_C_raw_response.txt"
RAW_RESPONSE_PATH.write_text(RAW_RESPONSE_TEXT, encoding="utf-8")
RAW_RESPONSE_SHA256 = sha256_file(RAW_RESPONSE_PATH)

print("Raw response preserved unchanged.")
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


Saving D2_branch_C_raw_response.txt to D2_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: 2b4410e60fb037b631990fe1ab190b48e8be73711f89593ef10781bf611827f8


In [27]:
# ------------------------------------------------------------
# 15. Parse the raw response without repair
# ------------------------------------------------------------
json_valid = True
json_error = None
parsed_extraction = None

try:
    parsed_extraction = json.loads(RAW_RESPONSE_TEXT)
except json.JSONDecodeError as exc:
    json_valid = False
    json_error = str(exc)

print("JSON valid:", json_valid)

if json_error:
    print("JSON parsing error:", json_error)


JSON valid: True


In [28]:
# ------------------------------------------------------------
# 16. Check top-level structure and record schema
# ------------------------------------------------------------
top_level_checks = {
    "output_is_json_object":
        isinstance(parsed_extraction, dict) if json_valid else False,
    "document_id_present":
        isinstance(parsed_extraction, dict)
        and "document_id" in parsed_extraction
        if json_valid else False,
    "document_id_correct":
        isinstance(parsed_extraction, dict)
        and parsed_extraction.get("document_id") == DOCUMENT_ID
        if json_valid else False,
    "branch_present":
        isinstance(parsed_extraction, dict)
        and "branch" in parsed_extraction
        if json_valid else False,
    "branch_correct":
        isinstance(parsed_extraction, dict)
        and parsed_extraction.get("branch") == BRANCH
        if json_valid else False,
    "records_present":
        isinstance(parsed_extraction, dict)
        and "records" in parsed_extraction
        if json_valid else False,
    "records_is_list":
        isinstance(parsed_extraction, dict)
        and isinstance(parsed_extraction.get("records"), list)
        if json_valid else False
}

records = (
    parsed_extraction.get("records", [])
    if json_valid and isinstance(parsed_extraction, dict)
    else []
)

observed_record_count = len(records) if isinstance(records, list) else None

record_structure_issues = []
field_type_issues = []
unit_issues = []

for idx, record in enumerate(records):

    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": idx,
            "issue": "record_is_not_json_object"
        })
        continue

    actual_fields = set(record.keys())
    expected_fields = set(EXPECTED_FIELDS)

    missing_fields = sorted(expected_fields - actual_fields)
    additional_fields = sorted(actual_fields - expected_fields)

    if missing_fields or additional_fields:
        record_structure_issues.append({
            "record_index": idx,
            "missing_fields": missing_fields,
            "additional_fields": additional_fields
        })

    line_item = record.get("Line Item")
    unit = record.get("Unit")

    if line_item is not None and not isinstance(line_item, str):
        field_type_issues.append({
            "record_index": idx,
            "field": "Line Item",
            "observed_type": type(line_item).__name__
        })

    if unit is not None and not isinstance(unit, str):
        field_type_issues.append({
            "record_index": idx,
            "field": "Unit",
            "observed_type": type(unit).__name__
        })

    for field in NUMERIC_FIELDS:
        value = record.get(field)
        if value is not None and (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
        ):
            field_type_issues.append({
                "record_index": idx,
                "field": field,
                "observed_type": type(value).__name__
            })

    if unit is not None and isinstance(unit, str) and unit not in ALLOWED_UNITS:
        unit_issues.append({
            "record_index": idx,
            "line_item": line_item,
            "observed_unit": unit
        })

records_with_type_issues = len({
    issue["record_index"]
    for issue in field_type_issues
})


In [29]:
# ------------------------------------------------------------
# 17. Keep schema validity separate from scope completeness
# ------------------------------------------------------------
# Record-count agreement is an extraction/scope outcome, not part of
# schema validity. This matches the dissertation's Stage 4 logic.

top_level_valid = all(top_level_checks.values())

schema_validity = bool(
    json_valid
    and top_level_valid
    and len(record_structure_issues) == 0
    and records_with_type_issues == 0
    and len(unit_issues) == 0
)

scope_complete = bool(
    observed_record_count == EXPECTED_RECORD_COUNT
    if observed_record_count is not None
    else False
)

line_items = [
    record.get("Line Item")
    for record in records
    if isinstance(record, dict)
]

duplicate_line_items = sorted({
    item
    for item in line_items
    if item is not None and line_items.count(item) > 1
})

missing_values_by_field = {
    field: sum(
        1
        for record in records
        if not isinstance(record, dict)
        or record.get(field) is None
    )
    for field in EXPECTED_FIELDS
}

structure_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "json_valid": json_valid,
    "json_error": json_error,
    "top_level_checks": top_level_checks,
    "schema_validity": schema_validity,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "scope_complete": scope_complete,
    "records_with_structure_issues": len(record_structure_issues),
    "record_structure_issues": record_structure_issues,
    "records_with_type_issues": records_with_type_issues,
    "field_type_issues": field_type_issues,
    "records_with_unexpected_units": len(unit_issues),
    "unit_issues": unit_issues,
    "duplicate_line_item_count": len(duplicate_line_items),
    "duplicate_line_items": duplicate_line_items,
    "missing_values_by_field": missing_values_by_field
}

STRUCTURE_CHECK_PATH = OUTPUT_DIR / "D2_branch_C_structure_check.json"
STRUCTURE_CHECK_PATH.write_text(
    json.dumps(structure_check, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(structure_check, indent=2, ensure_ascii=False))


{
  "document_id": "D2",
  "branch": "C",
  "json_valid": true,
  "json_error": null,
  "top_level_checks": {
    "output_is_json_object": true,
    "document_id_present": true,
    "document_id_correct": true,
    "branch_present": true,
    "branch_correct": true,
    "records_present": true,
    "records_is_list": true
  },
  "schema_validity": true,
  "expected_record_count": 22,
  "observed_record_count": 22,
  "scope_complete": true,
  "records_with_structure_issues": 0,
  "record_structure_issues": [],
  "records_with_type_issues": 0,
  "field_type_issues": [],
  "records_with_unexpected_units": 0,
  "unit_issues": [],
  "duplicate_line_item_count": 0,
  "duplicate_line_items": [],
  "missing_values_by_field": {
    "Line Item": 0,
    "Unit": 0,
    "Value 2024": 0,
    "Value 2023": 0
  }
}


In [30]:
# ------------------------------------------------------------
# 18. Preserve parsed extraction only when JSON is valid
# ------------------------------------------------------------
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D2_branch_C_parsed_extraction.json"

if json_valid:
    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(parsed_extraction, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("Parsed extraction saved:", PARSED_EXTRACTION_PATH.name)
else:
    print(
        "No parsed extraction was created because the preserved "
        "raw response is not valid JSON."
    )


Parsed extraction saved: D2_branch_C_parsed_extraction.json


In [31]:
# ------------------------------------------------------------
# 19. Create final Branch C experiment summary
# ------------------------------------------------------------
summary = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "reference_values_used_for_transformation": False,
    "expected_record_count_disclosed_to_model": False,
    "raw_response_preserved": True,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "json_valid": json_valid,
    "schema_validity": schema_validity,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "scope_complete": scope_complete,
    "records_with_structure_issues": len(record_structure_issues),
    "records_with_type_issues": records_with_type_issues,
    "records_with_unexpected_units": len(unit_issues),
    "duplicate_line_item_count": len(duplicate_line_items),
    "parsed_extraction_created": json_valid,
    "accuracy_validation_completed": False,
    "validation_status":
        "Pending Stage 4 Branch C validation against the fixed Stage 1 "
        "reference dataset using Branch A-frozen comparison rules"
}

SUMMARY_PATH = OUTPUT_DIR / "D2_branch_C_experiment_summary.json"
SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "document_id": "D2",
  "document_name": "VINCI Consolidated Income Statement 2024",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_sha256": "6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667",
  "source_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "representation_file": "D2_branch_C_normalised_markdown.md",
  "representation_sha256": "cc5ca221b7a5af52fe977137c5ae4f4773cc7d0243719ea18be80f4b4d609143",
  "reference_values_used_for_transformation": false,
  "expected_record_count_disclosed_to_model": false,
  "raw_response_preserved": true,
  "raw_response_sha256": "2b4410e60fb037b631990fe1ab190b48e8be73711f89593ef10781bf611827f8",
  "json_valid": true,
  "schema_validity": true,
  "expected_record_count": 22,
  "observed_record_count": 22,
  "scope_complete": true,
  "records_with_structure_issues": 0,
  "records_with_type_issues": 0,
  "records_with_unexpected_un

In [32]:
# ------------------------------------------------------------
# 20. Final artefact inventory and download
# ------------------------------------------------------------
artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    SUMMARY_PATH
]

if json_valid:
    artefacts.append(PARSED_EXTRACTION_PATH)

print("Final D2 Branch C artefacts:")
for p in artefacts:
    print("-", p.name, "| exists:", p.exists())

for p in artefacts:
    if p.exists():
        files.download(p)


Final D2 Branch C artefacts:
- D2_branch_C_parent_B_equivalence_check.json | exists: True
- D2_branch_C_normalisation_check.json | exists: True
- D2_branch_C_normalised_markdown.md | exists: True
- D2_branch_C_prompt.txt | exists: True
- D2_branch_C_representation_metadata.json | exists: True
- D2_branch_C_experiment_metadata.json | exists: True
- D2_branch_C_pre_extraction_check.json | exists: True
- D2_branch_C_raw_response.txt | exists: True
- D2_branch_C_structure_check.json | exists: True
- D2_branch_C_experiment_summary.json | exists: True
- D2_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>